# Portfolio consumption report — 2023 (fixed)

Corrected version of `mock_06`. Each fix is marked with **Fix N**, numbered as in
`mock_06_solution.md`.

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)

meters = pd.read_csv("../../data/meters.csv", parse_dates=["signup_date"])
readings = pd.read_csv("../../data/meter_readings_daily.csv", parse_dates=["date"])
meters.shape, readings.shape

((300, 7), (107503, 3))

**Fix 10** — normalise the region labels before any grouping (6 rows were lower-case).

In [2]:
print(meters["region"].value_counts().to_dict())
meters["region"] = meters["region"].str.strip().str.title()
print(meters["region"].value_counts().to_dict())

{'London': 96, 'North': 65, 'Scotland': 57, 'Midlands': 44, 'Wales': 32, 'london': 3, 'wales': 1, 'north': 1, 'midlands': 1}
{'London': 99, 'North': 66, 'Scotland': 57, 'Midlands': 45, 'Wales': 33}


**Fix 0 (merge hygiene)** — validate the key relationship and keep the merge indicator so orphan
readings are visible instead of becoming NaN-region rows.

In [3]:
data = readings.merge(meters, on="meter_id", how="left", validate="many_to_one", indicator=True)
print(data["_merge"].value_counts().to_dict())
orphans = data.loc[data["_merge"] == "left_only", "meter_id"].unique()
print("orphan meter ids:", orphans, "rows:", (data["_merge"] == "left_only").sum())
data = data.loc[data["_merge"] == "both"].drop(columns="_merge")
data["has_solar"] = data["has_solar"].astype(bool)   # object after the left merge
print("meters with no readings:", set(meters["meter_id"]) - set(data["meter_id"]))
data.shape

{'both': 107303, 'left_only': 200, 'right_only': 0}
orphan meter ids: ['M999999'] rows: 200
meters with no readings: set()


(107303, 9)

**Fix 1 / 2** — "meters per region" is a question about the meter table: `size()` on `meters`,
not on the merged reading-level frame, and not `count()` on a column with NaNs.

In [4]:
meters_per_region = meters.groupby("region").size().sort_values(ascending=False)
print("sum:", meters_per_region.sum())
print(pd.DataFrame({
    "size()": meters.groupby("region").size(),
    "tariff.count()": meters.groupby("region")["tariff"].count(),
}))

sum: 300
          size()  tariff.count()
region                          
London        99              93
Midlands      45              41
North         66              65
Scotland      57              55
Wales         33              33


**Fix 3** — `groupby` drops NaN keys by default. Use `dropna=False` so the 13 meters without a
tariff appear as their own segment, and check the table total against the grand total.

In [5]:
segment = data.groupby(["region", "tariff"], dropna=False)["kwh"].sum().unstack()
segment.columns = [c if isinstance(c, str) else "no_tariff" for c in segment.columns]
print(f"segment table total {segment.sum().sum():,.0f}  vs  grand total {data['kwh'].sum():,.0f}")
segment.round(0)

segment table total 1,692,372  vs  grand total 1,692,372


,Fixed,TOU,Variable,no_tariff
region,,,,
London,282334.0,109520.0,142779.0,52175.0
Midlands,65769.0,73176.0,108359.0,39268.0
North,171011.0,35681.0,159188.0,2525.0
Scotland,117101.0,22705.0,93561.0,4988.0
Wales,149200.0,17449.0,45583.0,NaN


**Fix 9** — `agg`, not `transform`, for a summary per customer type.

In [6]:
data.groupby("customer_type")["kwh"].agg(["mean", "median", "count"]).round(2)

,mean,median,count
customer_type,,,
residential,8.88,8.19,95131
sme,69.66,63.94,12172


**Fix 4** — solar penetration is a share of *meters*, so compute it on the meter table.

In [7]:
solar_share = meters.groupby("region")["has_solar"].mean().round(3)
pd.DataFrame({"meter-weighted": solar_share,
              "reading-weighted (mock)": data.groupby("region")["has_solar"].mean().round(3)})

,meter-weighted,reading-weighted (mock)
region,,
London,0.131,0.131
Midlands,0.022,0.022
North,0.136,0.136
Scotland,0.140,0.141
Wales,0.091,0.091


**Fix 12** — `normalize="index"` for shares *within* a region.

In [8]:
tariff_mix = pd.crosstab(meters["region"], meters["tariff"], normalize="index").round(3)
tariff_mix

tariff,Fixed,TOU,Variable
region,,,
London,0.505,0.172,0.323
Midlands,0.366,0.220,0.415
North,0.523,0.185,0.292
Scotland,0.545,0.145,0.309
Wales,0.515,0.152,0.333


**Fix 5** — top meters must be an aggregate per meter (here total kWh alongside mean kWh/day and the
number of days observed), not the 10 largest single daily readings.

In [9]:
per_meter = data.groupby("meter_id")["kwh"].agg(total="sum", per_day="mean", days="count")
top10 = per_meter.nlargest(10, "total").join(meters.set_index("meter_id")[["region", "customer_type"]])
top10.round(1)

,total,per_day,days,region,customer_type
meter_id,,,,,
M100015,37316.1,104.5,357,London,sme
M100240,36498.8,102.0,358,Midlands,sme
M100054,35238.7,98.7,357,North,sme
M100236,35105.3,98.1,358,London,sme
M100160,32999.4,91.9,359,North,sme
M100243,31886.8,87.6,364,Wales,sme
M100289,29799.5,84.4,353,Midlands,sme
M100107,29676.0,83.4,356,North,sme
M100117,29665.0,84.0,353,Scotland,sme


**Fix 6 / 8** — `aggfunc="sum"` for totals, and `pct_change()` down the month axis (axis=0).

In [10]:
data["month"] = data["date"].dt.to_period("M")
trend = data.pivot_table(index="month", columns="tariff", values="kwh", aggfunc="sum")
growth = trend.pct_change()          # axis=0: month over month
growth.round(3).head(6)

tariff,Fixed,TOU,Variable
month,,,
2023-01,NaN,NaN,NaN
2023-02,-0.127,-0.127,-0.123
2023-03,0.011,-0.002,0.002
2023-04,-0.164,-0.184,-0.176
2023-05,-0.158,-0.141,-0.143
2023-06,-0.198,-0.193,-0.191


**Fix 7** — with a single year of data a January-to-July fall is seasonality (heating), not a trend.
A trend claim needs the same month in different years, or a seasonally adjusted series.

In [11]:
monthly = data.groupby("month")["kwh"].sum()
(monthly / monthly.iloc[0] - 1).round(2).to_frame("vs_jan").T

month,2023-01,2023-02,2023-03,2023-04,2023-05,2023-06,2023-07,2023-08,2023-09,2023-10,2023-11,2023-12
vs_jan,0.0,-0.13,-0.12,-0.27,-0.38,-0.5,-0.52,-0.48,-0.4,-0.25,-0.15,-0.03


**Fix 11** — compare the annual estimate with a *full year* of actuals, scaled for the days actually
observed (2% of days are missing). Nine months of low season against an annual figure is not a
30% shortfall.

In [12]:
per_meter["annualised"] = per_meter["per_day"] * 365
cmp = meters.set_index("meter_id").join(per_meter)
cmp["ratio_full_year"] = cmp["annualised"] / cmp["annual_kwh_estimate"]
recent = data[data["date"] >= "2023-04-01"]
cmp["ratio_apr_dec_raw"] = recent.groupby("meter_id")["kwh"].sum() / cmp["annual_kwh_estimate"]
cmp[["ratio_full_year", "ratio_apr_dec_raw"]].describe().round(3)

,ratio_full_year,ratio_apr_dec_raw
count,294.000,294.000
mean,1.031,0.691
std,0.014,0.015
min,0.987,0.625
25%,1.021,0.683
50%,1.031,0.692
75%,1.041,0.701
max,1.071,0.730


## Results (honest)

In [13]:
print(f"1. Largest region: {meters_per_region.index[0]} with {meters_per_region.iloc[0]} meters "
      f"({meters_per_region.iloc[0] / meters_per_region.sum():.0%} of {meters_per_region.sum()})")
print(f"2. Portfolio consumption 2023: {data['kwh'].sum():,.0f} kWh (segment table {segment.sum().sum():,.0f}); "
      f"orphan meter rows excluded: {len(readings) - len(data)}")
print(f"3. Solar penetration: highest {solar_share.idxmax()} ({solar_share.max():.1%}), lowest {solar_share.idxmin()} ({solar_share.min():.1%})")
print(f"4. TOU share within London: {tariff_mix.loc['London', 'TOU']:.1%}")
print(f"5. Jan->Jul change {(monthly.iloc[6] / monthly.iloc[0] - 1):+.0%} is seasonal; no trend claim possible from one year")
print(f"6. Annualised actual vs estimate: {(cmp['ratio_full_year'].mean() - 1):+.1%} (Apr-Dec raw sum would say {(cmp['ratio_apr_dec_raw'].mean() - 1):+.0%})")

1. Largest region: London with 99 meters (33% of 300)
2. Portfolio consumption 2023: 1,692,372 kWh (segment table 1,692,372); orphan meter rows excluded: 200
3. Solar penetration: highest Scotland (14.0%), lowest Midlands (2.2%)
4. TOU share within London: 17.2%
5. Jan->Jul change -52% is seasonal; no trend claim possible from one year
6. Annualised actual vs estimate: +3.1% (Apr-Dec raw sum would say -31%)
